# Train your own StarDist-3D model

This script is a copy of the example notebook available in the [StarDist repository](https://github.com/stardist/stardist/tree/main) for training a 3D model. The parameters in these notebook are the ones that were used for training the models published in [Ferme et al 2024](https://www.biorxiv.org/content/10.1101/2024.11.12.623216v1). 

In [ ]:
from __future__ import print_function, unicode_literals, absolute_import, division
import sys
import os
import numpy as np
import pandas as pd
import csv
import matplotlib
import matplotlib.pyplot as plt

import subprocess
import datetime
import time
import warnings
import shutil
from skimage import io
from glob import glob
from tqdm import tqdm
from tifffile import imread, imsave
from pip._internal.operations.freeze import freeze

import tensorflow as tf
from csbdeep.utils import Path, normalize, download_and_extract_zip_file, plot_history # for loss plot
from csbdeep.io import save_tiff_imagej_compatible
from stardist import fill_label_holes, random_label_cmap, calculate_extents, gputools_available
from stardist import Rays_GoldenSpiral
from stardist.matching import matching, matching_dataset
from stardist.models import Config3D, StarDist3D, StarDistData3D
from augmend import Augmend, Elastic, Identity, FlipRot90, AdditiveNoise, CutOut, IsotropicScale, GaussianBlur, Rotate, IntensityScaleShift, CutOut, Choice, Rotate, BaseTransform
%load_ext tensorboard

# function for plotting validation and training datasets
def plot_img_label(img, lbl, img_title="image (XY slice)", lbl_title="label (XY slice)", name=None, z=None, **kwargs):
    plt.style.use("dark_background")
    if z is None:
        z = img.shape[0] // 2    
    fig, (ai,al) = plt.subplots(1,2, figsize=(12,5), gridspec_kw=dict(width_ratios=(1.25,1)))
    im = ai.imshow(img[z], cmap='gray', clim=(0,1))
    ai.set_title(img_title)    
    fig.colorbar(im, ax=ai)
    al.imshow(lbl[z], cmap=lbl_cmap)
    al.set_title(lbl_title)
    plt.tight_layout()
    if name != None and type(name==str):
        plt.savefig(model_dir+"/"+name)

### Set up the parameters for training

In [ ]:
### Path to .czi files and to output folder
inputdir = '...'  ### args.pathname
#outputdir = os.path.join(inputdir, "images")

# set up a colormap 
np.random.seed(42)
bl_cmap = random_label_cmap()


# define the path where the model will be saved
version = ""
model_path = "models"
model_name = "model_"+version
model_dir = os.path.join(inputdir, "/"+model_path+"/"+model_name)
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

# Training parameters for the config
initial_learning_rate = 0.003
Use_Data_augmentation = True
n_rays = 256
number_of_epochs = 300
patch_height = 32
patch_size = 128
batch_size = 2
number_of_steps = 100
grid = (1,4,4)

print("Model Directory ", model_dir, "\n")

### Configure the model parameters 

In [ ]:
X = sorted(glob(os.path.join(inputdir, 'train/images/*.tif')))
Y = sorted(glob(os.path.join(pathname,'train/masks/*.tif')))

# Control that you have same number of images and masks
print(f'found {len(X)} training images and {len(Y)} masks')
assert all(Path(x).name == Path(y).name for x,y in zip(X,Y))

# imread returns image data from .tif as a numpy array
X = list(map(imread,X))
Y = list(map(imread,Y))
Y = [np.array(y, dtype='int') for y in Y]
n_channel = 1 if X[0].ndim == 3 else X[0].shape[-1]
print("\nShape of the first image: ", X[0].shape)


axis_norm = (0,1,2)   # normalize channels independently
# axis_norm = (0,1,2,3) # normalize channels jointly
if n_channel > 1:
    print("\nNormalizing image channels %s." % ('jointly' if axis_norm is None or 3 in axis_norm else 'independently'))
    sys.stdout.flush()

# Normalize images
X = [normalize(x,1,99.8,axis=axis_norm) for x in tqdm(X)]
# Fill holes in the masked
Y = [fill_label_holes(y) for y in tqdm(Y)]
assert len(X) > 1, "not enough training data"

# Split the datasets into training and validation sets
rng = np.random.RandomState(42)
ind = rng.permutation(len(X))
n_val = max(1, int(round(0.15 * len(ind))))
ind_train, ind_val = ind[:-n_val], ind[-n_val:]
X_val, Y_val = [X[i] for i in ind_val]  , [Y[i] for i in ind_val]
X_trn, Y_trn = [X[i] for i in ind_train], [Y[i] for i in ind_train]
print('number of images: %3d' % len(X))
print('- training:       %3d' % len(X_trn))
print('- validation:     %3d' % len(X_val))


print("Selected patch size:", patch_height, patch_size, patch_size)
patch = (patch_height, patch_size, patch_size)
for x,y in zip(X,Y):
    print(y.ndim, y.ndim==len(patch))
    print(X[0].ndim, x.ndim==X[0].ndim) #x_ndim
    print(x.shape[:len(patch)], y.shape, x.shape[:len(patch)]==y.shape ) 

# plot one example for validation and training images
i = 0
img, lbl = X_val[i], Y_val[i]
assert img.ndim in (3,4)
img = img if img.ndim==3 else img[...,:3]
plot_img_label(img,lbl, name="ValidationDataExample_StarDist3D")

i = 0
img, lbl = X_trn[i], Y_trn[i]
assert img.ndim in (3,4)
img = img if img.ndim==3 else img[...,:3]
plot_img_label(img,lbl, name="TrainingDataExample_StarDist3D")


# Now calculate the anisotropy on the training data
extents = calculate_extents(Y)
anisotropy = tuple(np.max(extents) / extents)
print('\nEmpirical anisotropy of labeled objects = %s' % str(anisotropy))


# Use OpenCL-based computations for data generator during training (requires 'gputools')
use_gpu = True and gputools_available()

# Use rays on a Fibonacci lattice adjusted for measured anisotropy of the training data
rays = Rays_GoldenSpiral(n_rays, anisotropy=anisotropy)

# configure the model parameters
conf = Config3D (
    rays             = rays,
    grid             = grid,
    anisotropy       = anisotropy,
    backbone         = "unet",
    unet_pool        = (2,4,4),
    use_gpu          = use_gpu,
    n_channel_in     = n_channel,
    train_learning_rate = initial_learning_rate,
    # adjust for your data below (make patch size as large as possible)
    train_patch_size = (patch_height, patch_size, patch_size), # v2.21 = 48, 96, 96
    train_batch_size = batch_size,
    train_epochs     = number_of_epochs,
    train_steps_per_epoch = number_of_steps,
    train_tensorboard = True
)
vars(conf)


# call GPU before doing any computation (such as calculating the fov)
if use_gpu:
    from csbdeep.utils.tf import limit_gpu_memory
    # limit GPU memory to be used by TensorFlow to leave some to OpenCL-based computations
    limit_gpu_memory(0.8, total_memory=48000)
    

# create the model
model = StarDist3D(conf, name="", basedir=os.path.join(model_path,model_name)) #name="Lucrezia/stardist", 

# Calculate the median object size and field of view
median_size = calculate_extents(Y, np.median)
fov = np.array(model._axes_tile_overlap('ZYX'))
print(f"\nmedian object size:      {median_size}")
print(f"network field of view :  {fov}")
if any(median_size > fov):
    print("WARNING: median object size larger than field of view of the neural network.")


### Define augmentation function

In [ ]:

aug = Augmend()
aug.add([FlipRot90(axis=(1)),FlipRot90(axis=(1))])
aug.add([Rotate(axis = (1,2), order=0), Rotate(axis = (1,2), order=0)])
aug.add([Elastic(grid=5, amount=12, order=0, axis = (2), use_gpu=True),
         Elastic(grid=5, amount=12, order=0, axis = (2), use_gpu=True)]),
aug.add([IsotropicScale(axis = (1,2), amount=(.8,1.2), order=0),
        IsotropicScale(axis = (1,2), amount=(.8,1.2), order=0)])
aug.add([AdditiveNoise(sigma=(0,.05)),Identity()])
aug.add([IntensityScaleShift(scale=(.5,2), shift=(-.2,.2)),Identity()])
   
   

def augmenter(x,y):
    return aug([x,y])


### Training and optimization of the model

In [ ]:
start = time.time()

history = model.train(X_trn, Y_trn, validation_data=(X_val,Y_val), augmenter=augmenter)
print("Training done")
%tensorboard --logdir logs/fit 


print("Network optimization in progress")

#Optimize the network.
model.optimize_thresholds(X_val, Y_val, iou_threshs=[0.1, 0.2, 0.4, 0.5, 0.7])


# Displaying the time elapsed for training
dt = time.time() - start
mins, sec = divmod(dt, 60)
hour, mins = divmod(mins, 60)
print("Time elapsed:",hour, "hour(s)",mins,"min(s)",round(sec),"sec(s)")

### Export your model into the BioImage Model Zoo format

In [ ]:
import os
from tifffile import imread
from stardist import export_bioimageio, import_bioimageio
from stardist.bioimageio_utils import _get_stardist_metadata
from stardist.models import StarDist3D
from csbdeep.utils import Path
import numpy as np


In [ ]:
### Define the path where the model you want to use is located
version = "" #name of the model you want to save
model_name = "model_v"+version
model_path = "C:/Users/lcferme/Desktop/stardist/models"
print("Model Directory ", model_path, "\n")

In [ ]:
# Provide the path to an example input file to be exported with the model 
fileID    =  "C:/Users/lcferme/Desktop/stardist/example_input/20220725_30hpf_H2B-RFP_Ath5-GFP_E2_test-128.tif"

example_img = imread(fileID)


# Load the model
model = StarDist3D(None, name=model_name, basedir=model_path)
modelfile = Path(os.path.join(model_path, model_name + '_bioimageio.zip'))

## The following functions should be provisional until StarDist+CSBDeep migrate
# completely to TF2
def _is_power_of_2(i):
    assert i > 0
    e = np.log2(i)
    return e == int(e)

def _my_compute_receptive_field(model, img_size=None):
    # TODO: good enough?
    from scipy.ndimage import zoom
    if img_size is None:
        img_size = tuple(g*(128 if model.config.n_dim==2 else 64) for g in model.config.grid)
    if np.isscalar(img_size):
        img_size = (img_size,) * model.config.n_dim
    img_size = tuple(img_size)
    # print(img_size)
    assert all(_is_power_of_2(s) for s in img_size)
    mid = tuple(s//2 for s in img_size)
    x = np.zeros((1,)+img_size+(model.config.n_channel_in,), dtype=np.float32)
    z = np.zeros_like(x)
    x[(0,)+mid+(slice(None),)] = 1
    #y  = self.keras_model.predict(x)[0][0,...,0]
    #y0 = self.keras_model.predict(z)[0][0,...,0]
    y  = model.predict(np.squeeze(x))[0]
    y0 = model.predict(np.squeeze(z))[0]

    grid = tuple((np.array(x.shape[1:-1])/np.array(y.shape)).astype(int))
    assert grid == model.config.grid
    y  = zoom(y, grid,order=0)
    y0 = zoom(y0,grid,order=0)
    ind = np.where(np.abs(y-y0)>0)
    return [(m-np.min(i), np.max(i)-m) for (m,i) in zip(mid,ind)]
    
model._tile_overlap = _my_compute_receptive_field(model) 
print('Comput_receptive field', _my_compute_receptive_field(model))
export_bioimageio(model, modelfile, example_img)